In [49]:
import pandas as pd
import os

# ---------------------------------------------------------
# 1. RUTAS Y CARGA DE DATOS
# ---------------------------------------------------------

carpeta = r"C:\Users\angel\OneDrive\Documents\Curso practicas\Proyecto fin de Master"

archivo_wb = os.path.join(carpeta, "259a748e-6760-43af-b0c6-435f5205e8d4.csv")
url_energy = "https://raw.githubusercontent.com/owid/energy-data/master/owid-energy-data.csv"

print("Cargando datos...")
df_wb = pd.read_csv(archivo_wb)
df_energy = pd.read_csv(url_energy)
print("Datos cargados correctamente.\n")

# ---------------------------------------------------------
# 2. LIMPIEZA Y FORMATO LARGO DEL WORLD BANK
# ---------------------------------------------------------

if "Time" in df_wb.columns:
    df_wb = df_wb.rename(columns={"Time": "year"})

df_wb["year"] = pd.to_numeric(df_wb["year"], errors="coerce")
df_wb = df_wb.dropna(subset=["Country Name", "Country Code", "year"]).copy()

meta_cols = ["Country Name", "Country Code", "year", "Time Code"]
indicator_cols = [c for c in df_wb.columns if c not in meta_cols]

rows = []

for col in indicator_cols:
    col_str = str(col)

    if "[" in col_str:
        indicator_name = col_str.split("[")[0].strip()
    else:
        indicator_name = col_str.strip()

    if "[" in col_str and "]" in col_str:
        start = col_str.find("[") + 1
        end = col_str.find("]")
        indicator_code = col_str[start:end].strip()
    else:
        indicator_code = col_str.strip()

    tmp = df_wb[["Country Name", "Country Code", "year", col]].copy()
    tmp = tmp.rename(columns={col: "value"})
    tmp["Indicator Name"] = indicator_name
    tmp["Indicator Code"] = indicator_code

    rows.append(tmp)

df_wb_long = pd.concat(rows, ignore_index=True)

df_wb_long["year"] = pd.to_numeric(df_wb_long["year"], errors="coerce")
df_wb_long["value"] = pd.to_numeric(df_wb_long["value"], errors="coerce")
df_wb_long = df_wb_long.dropna(subset=["year", "value"]).copy()

df_wb_long = df_wb_long[
    (df_wb_long["year"] >= 1990) & (df_wb_long["year"] <= 2024)
].copy()

# ---------------------------------------------------------
# 3. FILTRAR SOLO LOS INDICADORES ECONÓMICOS QUE PEDISTE
# ---------------------------------------------------------

indicadores_economia = [
    "FP.CPI.TOTL.ZG",
    "NY.GDP.PCAP.KD",
    "NY.GDP.MKTP.KD.ZG",
    "SL.UEM.TOTL.ZS",
    "NY.GNP.PCAP.CD",
    "SL.TLF.CACT.ZS",
    "SP.POP.TOTL",
    "SP.URB.TOTL.IN.ZS",
    "SE.TER.ENRR"
]

df_wb_filtered = df_wb_long[df_wb_long["Indicator Code"].isin(indicadores_economia)].copy()

# ---------------------------------------------------------
# 4. LIMPIEZA DEL OWID ENERGY
# ---------------------------------------------------------

df_energy = df_energy.rename(columns={
    "country": "Country Name",
    "iso_code": "Country Code",
    "year": "year"
})

df_energy["year"] = pd.to_numeric(df_energy["year"], errors="coerce")
df_energy = df_energy.dropna(subset=["Country Name", "Country Code", "year"]).copy()

df_energy = df_energy[
    (df_energy["year"] >= 1990) & (df_energy["year"] <= 2024)
].copy()

energy_cols = [
    "energy_cons_change_pct",
    "energy_per_capita",
    "fossil_share_energy",
    "renewables_share_energy",
    "solar_share_energy",
    "wind_share_energy",
    "coal_share_energy",
    "oil_share_energy",
    "gas_share_energy",
    "biofuel_share_energy",
    "low_carbon_share_energy"
]

df_energy_sel = df_energy[
    ["Country Name", "Country Code", "year"] + energy_cols
].copy()

# ---------------------------------------------------------
# 5. UNIÓN SOLO DE PAÍSES Y AÑOS COMUNES (INNER)
# ---------------------------------------------------------

df_final = pd.merge(
    df_wb_filtered,
    df_energy_sel,
    on=["Country Name", "Country Code", "year"],
    how="inner"
)

# ---------------------------------------------------------
# 6. CREAR NUEVO INDICADOR EN FORMATO LARGO
# ---------------------------------------------------------

# Extraer población
df_pop = df_final[df_final["Indicator Code"] == "SP.POP.TOTL"][["Country Name", "Country Code", "year", "value"]]
df_pop = df_pop.rename(columns={"value": "population_wb"})

# Extraer escolarización terciaria (%)
df_ter = df_final[df_final["Indicator Code"] == "SE.TER.ENRR"][["Country Name", "Country Code", "year", "value"]]
df_ter = df_ter.rename(columns={"value": "tertiary_pct"})

# Merge para calcular el indicador derivado
df_derived = pd.merge(df_pop, df_ter, on=["Country Name", "Country Code", "year"], how="inner")

# Calcular valor
df_derived["value"] = df_derived["population_wb"] * (df_derived["tertiary_pct"] / 100)

# Crear filas nuevas en formato largo
df_derived["Indicator Name"] = "School enrollment, tertiary (Total)"
df_derived["Indicator Code"] = "SE.TER.ENRR.TOTL"

# Añadir columnas energéticas
df_derived = pd.merge(
    df_derived,
    df_energy_sel,
    on=["Country Name", "Country Code", "year"],
    how="left"
)

# Añadir al dataset final
df_final = pd.concat([df_final, df_derived], ignore_index=True)

# ---------------------------------------------------------
# 7. CATEGORÍAS ADICIONALES (SE MANTIENEN)
# ---------------------------------------------------------

def safe_qcut(series, labels):
    try:
        return pd.qcut(series, q=4, labels=labels, duplicates="drop")
    except:
        return pd.Series([None] * len(series))

# GDP category
df_gdp = df_final[df_final["Indicator Code"] == "NY.GDP.PCAP.KD"]
df_gdp_cat = safe_qcut(df_gdp["value"], ["GDP_bajo", "GDP_medio_bajo", "GDP_medio_alto", "GDP_alto"])
df_gdp["GDP_category"] = df_gdp_cat
df_final = df_final.merge(df_gdp[["Country Name", "year", "GDP_category"]], on=["Country Name", "year"], how="left")

# Population category
df_pop_cat = safe_qcut(df_pop["population_wb"], ["Poblacion_baja", "Poblacion_media_baja", "Poblacion_media_alta", "Poblacion_alta"])
df_pop["Population_category"] = df_pop_cat
df_final = df_final.merge(df_pop[["Country Name", "year", "Population_category"]], on=["Country Name", "year"], how="left")

# Renewables category
df_ren = df_energy_sel.copy()
df_ren_cat = safe_qcut(df_ren["renewables_share_energy"], ["Renovables_bajas", "Renovables_medias_bajas", "Renovables_medias_altas", "Renovables_altas"])
df_ren["Renewables_category"] = df_ren_cat
df_final = df_final.merge(df_ren[["Country Name", "year", "Renewables_category"]], on=["Country Name", "year"], how="left")

# ---------------------------------------------------------
# 7. CATEGORÍAS ADICIONALES (TODAS)
# ---------------------------------------------------------

def safe_qcut(series, labels):
    try:
        return pd.qcut(series, q=4, labels=labels, duplicates="drop")
    except:
        return pd.Series([None] * len(series))

# GDP category
df_gdp = df_final[df_final["Indicator Code"] == "NY.GDP.PCAP.KD"]
df_gdp_cat = safe_qcut(df_gdp["value"], ["GDP_bajo", "GDP_medio_bajo", "GDP_medio_alto", "GDP_alto"])
df_gdp["GDP_category"] = df_gdp_cat
df_final = df_final.merge(df_gdp[["Country Name", "year", "GDP_category"]], on=["Country Name", "year"], how="left")

# Population category
df_pop_cat = safe_qcut(df_pop["population_wb"], ["Poblacion_baja", "Poblacion_media_baja", "Poblacion_media_alta", "Poblacion_alta"])
df_pop["Population_category"] = df_pop_cat
df_final = df_final.merge(df_pop[["Country Name", "year", "Population_category"]], on=["Country Name", "year"], how="left")

# Renewables category
df_ren = df_energy_sel.copy()
df_ren_cat = safe_qcut(df_ren["renewables_share_energy"], ["Renovables_bajas", "Renovables_medias_bajas", "Renovables_medias_altas", "Renovables_altas"])
df_ren["Renewables_category"] = df_ren_cat
df_final = df_final.merge(df_ren[["Country Name", "year", "Renewables_category"]], on=["Country Name", "year"], how="left")

# NUEVAS CATEGORÍAS SOCIOECONÓMICAS

# Educación terciaria
df_edu = df_final[df_final["Indicator Code"] == "SE.TER.ENRR"][["Country Name", "year", "value"]].copy()
df_edu_cat = safe_qcut(df_edu["value"], ["Edu_baja", "Edu_media_baja", "Edu_media_alta", "Edu_alta"])
df_edu["Education_category"] = df_edu_cat
df_final = df_final.merge(df_edu[["Country Name", "year", "Education_category"]], on=["Country Name", "year"], how="left")

# Urbanización
df_urb = df_final[df_final["Indicator Code"] == "SP.URB.TOTL.IN.ZS"][["Country Name", "year", "value"]].copy()
df_urb_cat = safe_qcut(df_urb["value"], ["Urb_baja", "Urb_media_baja", "Urb_media_alta", "Urb_alta"])
df_urb["Urban_category"] = df_urb_cat
df_final = df_final.merge(df_urb[["Country Name", "year", "Urban_category"]], on=["Country Name", "year"], how="left")

# Desempleo
df_unemp = df_final[df_final["Indicator Code"] == "SL.UEM.TOTL.ZS"][["Country Name", "year", "value"]].copy()
df_unemp_cat = safe_qcut(df_unemp["value"], ["Desemp_bajo", "Desemp_medio_bajo", "Desemp_medio_alto", "Desemp_alto"])
df_unemp["Unemployment_category"] = df_unemp_cat
df_final = df_final.merge(df_unemp[["Country Name", "year", "Unemployment_category"]], on=["Country Name", "year"], how="left")

# Participación laboral
df_labor = df_final[df_final["Indicator Code"] == "SL.TLF.CACT.ZS"][["Country Name", "year", "value"]].copy()
df_labor_cat = safe_qcut(df_labor["value"], ["Labor_baja", "Labor_media_baja", "Labor_media_alta", "Labor_alta"])
df_labor["LaborForce_category"] = df_labor_cat
df_final = df_final.merge(df_labor[["Country Name", "year", "LaborForce_category"]], on=["Country Name", "year"], how="left")

# PIB per cápita constante 2015
df_gdp2015 = df_final[df_final["Indicator Code"] == "NY.GDP.PCAP.KD"][["Country Name", "year", "value"]].copy()
df_gdp2015_cat = safe_qcut(df_gdp2015["value"], ["GDPpc_bajo", "GDPpc_media_baja", "GDPpc_media_alta", "GDPpc_alto"])
df_gdp2015["GDPpc2015_category"] = df_gdp2015_cat
df_final = df_final.merge(df_gdp2015[["Country Name", "year", "GDPpc2015_category"]], on=["Country Name", "year"], how="left")



# ---------------------------------------------------------
# 8. COMPROBAR TAMAÑO FINAL
# ---------------------------------------------------------

print("Filas finales:", len(df_final))
print("Columnas finales:", len(df_final.columns))

# ---------------------------------------------------------
# 9. GUARDAR ARCHIVO FINAL
# ---------------------------------------------------------

archivo_salida = os.path.join(carpeta, "dataset_final_economia_energia_largo_con_tertiary_total_indicator.csv")

df_final.to_excel("dataset_final.xlsx", index=False)

print("\nArchivo final generado correctamente:")
print(archivo_salida)


Cargando datos...
Datos cargados correctamente.

Filas finales: 51712
Columnas finales: 30

Archivo final generado correctamente:
C:\Users\angel\OneDrive\Documents\Curso practicas\Proyecto fin de Master\dataset_final_economia_energia_largo_con_tertiary_total_indicator.csv


In [ ]:
# ============================================================
#  ANALYSIS PIPELINE – ECONOMIC GROWTH, SOCIAL DEVELOPMENT & ENERGY TRANSITION
#  Proyecto de Data Analytics – Autor: A
# ============================================================

import pandas as pd
import numpy as np
import statsmodels.api as sm
from pathlib import Path
import os

# ------------------------------------------------------------
# 1. CONFIGURACIÓN DE ARCHIVOS
# ------------------------------------------------------------

carpeta = r"C:\Users\angel\OneDrive\Documents\Curso practicas\Proyecto fin de Master"
INPUT_FILE = "dataset_final.xlsx"
OUTPUT_FILE = "analysis_output.xlsx"

# ------------------------------------------------------------
# 2. CARGA DEL DATASET FINAL
# ------------------------------------------------------------

df_final = pd.read_excel(Path(carpeta) / INPUT_FILE)

# ------------------------------------------------------------
# 3. SELECCIÓN DE COLUMNAS ENERGÉTICAS EXTENDIDAS
# ------------------------------------------------------------

energy_cols_extended = [
    "energy_per_capita",
    "energy_cons_change_pct",
    "fossil_share_energy",
    "renewables_share_energy",
    "solar_share_energy",
    "wind_share_energy",
    "hydro_share_energy",
    "biofuel_share_energy",
    "nuclear_share_energy",
    "low_carbon_share_energy"
]

# Filtrar columnas energéticas si existen
energy_available = [c for c in energy_cols_extended if c in df_final.columns]

# ------------------------------------------------------------
# 4. RENOMBRAR COLUMNAS WB
# ------------------------------------------------------------

rename_map = {
    "Country Name": "country",
    "NY.GDP.PCAP.KD": "gdp_per_capita_constant_2015",
    "SE.TER.ENRR": "school_enrollment_tertiary_pct",
    "SP.URB.TOTL.IN.ZS": "urban_population_pct",
    "SL.UEM.TOTL.ZS": "unemployment_rate",
    "SL.TLF.CACT.ZS": "labor_force_participation_rate",
    "renewables_share_energy": "renewable_share"
}

df_final = df_final.rename(columns=rename_map)

# PIB alias
df_final = df_final.rename(columns=rename_map)

# PIB alias
# Crear/normalizar la columna del PIB per cápita real
if "gdp_per_capita_constant_2015" not in df_final.columns:
    if "NY.GDP.PCAP.KD" in df_final.columns:
        df_final = df_final.rename(columns={"NY.GDP.PCAP.KD": "gdp_per_capita_constant_2015"})
    elif "Indicator Code" in df_final.columns and "value" in df_final.columns:
        # Tras el renombrado inicial, "Country Name" puede llamarse "country".
        # Se crea un alias temporal para mantener compatibles las claves del merge.
        if "Country Name" not in df_final.columns and "country" in df_final.columns:
            df_final["Country Name"] = df_final["country"]

        gdp_aux = (
            df_final.loc[
            df_final["Indicator Code"].eq("NY.GDP.PCAP.KD"),
            ["Country Name", "Country Code", "year", "value"]
            ]
            .rename(columns={"value": "gdp_per_capita_constant_2015"})
            .drop_duplicates(["Country Name", "Country Code", "year"])
        )
        df_final = df_final.merge(gdp_aux, on=["Country Name", "Country Code", "year"], how="left")
    else:
        raise KeyError("No se encontró 'gdp_per_capita_constant_2015' ni 'NY.GDP.PCAP.KD' en el DataFrame.")

# Alias para compatibilidad con el resto del notebook
if "gdp" not in df_final.columns:
    df_final["gdp"] = df_final["gdp_per_capita_constant_2015"]

# ------------------------------------------------------------
# 5. MÉTRICAS ENERGÉTICAS DERIVADAS
# ------------------------------------------------------------

df_final["total_energy_consumption"] = (
    df_final["energy_per_capita"] * df_final["population_wb"] / 1_000_000
)

# ------------------------------------------------------------
# 6. FILTRAR PERIODO Y ORDENAR
# ------------------------------------------------------------

df = df_final.copy()
df = df[(df["year"] >= 1990) & (df["year"] <= 2024)]
df = df.sort_values(["country", "year"])

# ------------------------------------------------------------
# 7. MÉTRICAS DERIVADAS ECONÓMICAS Y SOCIALES
# ------------------------------------------------------------

df["gdp_growth_pct"] = df.groupby("country")["gdp"].pct_change() * 100
df["renewable_change_pct"] = df.groupby("country")["renewable_share"].pct_change()
df["renewable_change_pp"] = df.groupby("country")["renewable_share"].diff()

# Normalizar el indicador de educación terciaria.
# En dataset_final.xlsx aparece como `tertiary_pct`.
if "school_enrollment_tertiary_pct" not in df.columns:
    education_sources = [
        "tertiary_pct",
        "SE.TER.ENRR",
        "School enrollment, tertiary (% gross) [SE.TER.ENRR]"
    ]
    
    source = next((c for c in education_sources if c in df.columns), None)
    
    if source is None:
        raise KeyError(
            "No se encontró ninguna columna de educación terciaria. "
            f"Columnas disponibles: {df.columns.tolist()}"
        )
    
    df["school_enrollment_tertiary_pct"] = pd.to_numeric(
        df[source], errors="coerce"
    )

df["education_growth_pct"] = (
    df.groupby("country")["school_enrollment_tertiary_pct"]
      .pct_change() * 100
)
# Asegurar que exista la columna de urbanización
if "urban_population_pct" not in df.columns:
    urban_candidates = [
        "SP.URB.TOTL.IN.ZS",
        "Urban population (% of total population) [SP.URB.TOTL.IN.ZS]",
        "urban_population_pct",
        "urban_pct"
    ]
    urban_source = next((c for c in urban_candidates if c in df.columns), None)

    if urban_source is not None:
        df["urban_population_pct"] = pd.to_numeric(df[urban_source], errors="coerce")
    elif "df_urb" in globals() and isinstance(df_urb, pd.DataFrame):
        urb = (
            df_urb.rename(columns={"Country Name": "country", "value": "urban_population_pct"})
            [["country", "year", "urban_population_pct"]]
            .drop_duplicates()
        )
        df = df.merge(urb, on=["country", "year"], how="left")
    else:
        raise KeyError("No se encontró 'urban_population_pct' ni un origen alternativo en el DataFrame.")

df["urban_growth_pct"] = df.groupby("country")["urban_population_pct"].pct_change() * 100
# Asegurar que exista la columna de desempleo antes de calcular su variación
if "unemployment_rate" not in df.columns:
    unemployment_candidates = [
        "SL.UEM.TOTL.ZS",
        "Unemployment, total (% of total labor force) (modeled ILO estimate) [SL.UEM.TOTL.ZS]"
    ]
    unemployment_source = next(
        (c for c in unemployment_candidates if c in df.columns),
        None
    )

    if unemployment_source is not None:
        df["unemployment_rate"] = pd.to_numeric(
            df[unemployment_source], errors="coerce"
        )
    elif "df_unemp" in globals() and isinstance(df_unemp, pd.DataFrame):
        unemp = (
            df_unemp.rename(columns={
                "Country Name": "country",
                "value": "unemployment_rate"
            })
            [["country", "year", "unemployment_rate"]]
            .drop_duplicates(["country", "year"])
        )

        df = df.merge(unemp, on=["country", "year"], how="left")
        df["unemployment_rate"] = pd.to_numeric(
            df["unemployment_rate"], errors="coerce"
        )
    else:
        raise KeyError(
            "No se encontró ninguna fuente para 'unemployment_rate'."
        )

df["unemployment_change_pct"] = (
    df.groupby("country")["unemployment_rate"].pct_change() * 100
)
# Asegurar que exista la columna de participación laboral antes de calcular su variación
if "labor_force_participation_rate" not in df.columns:
    labor_candidates = [
        "SL.TLF.CACT.ZS",
        "Labor force participation rate, total (% of total population ages 15+) (modeled ILO estimate) [SL.TLF.CACT.ZS]"
    ]
    labor_source = next(
        (c for c in labor_candidates if c in df.columns),
        None
    )

    if labor_source is not None:
        df["labor_force_participation_rate"] = pd.to_numeric(
            df[labor_source], errors="coerce"
        )
    elif "df_labor" in globals() and isinstance(df_labor, pd.DataFrame):
        labor = (
            df_labor.rename(columns={
                "Country Name": "country",
                "value": "labor_force_participation_rate"
            })
            [["country", "year", "labor_force_participation_rate"]]
            .drop_duplicates(["country", "year"])
        )

        df = df.merge(labor, on=["country", "year"], how="left")
        df["labor_force_participation_rate"] = pd.to_numeric(
            df["labor_force_participation_rate"], errors="coerce"
        )
    else:
        raise KeyError(
            "No se encontró ninguna fuente para 'labor_force_participation_rate'."
        )

df["labor_force_change_pct"] = df.groupby("country")["labor_force_participation_rate"].pct_change() * 100
df["gdp_pc2015_growth_pct"] = df.groupby("country")["gdp_per_capita_constant_2015"].pct_change() * 100

# ------------------------------------------------------------
# 8. ÍNDICES COMPUESTOS
# ------------------------------------------------------------

df["SDI"] = (
    df["school_enrollment_tertiary_pct"].rank(pct=True) +
    df["gdp_per_capita_constant_2015"].rank(pct=True)
) / 2

df["UMI"] = (
    df["urban_population_pct"].rank(pct=True) +
    df["labor_force_participation_rate"].rank(pct=True)
) / 2

df["LEI"] = (
    df["labor_force_participation_rate"].rank(pct=True) -
    df["unemployment_rate"].rank(pct=True)
)

# ------------------------------------------------------------
# 9. CLUSTERING SOCIOECONÓMICO + TRANSICIÓN ENERGÉTICA
# ------------------------------------------------------------

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.impute import SimpleImputer

cluster_vars = [
    "gdp_per_capita_constant_2015",
    "school_enrollment_tertiary_pct",
    "urban_population_pct",
    "labor_force_participation_rate",
    "unemployment_rate",
    "SDI",
    "UMI",
    "LEI",
    "renewable_share",
    "renewable_change_pct",
    "total_energy_consumption",
    "gdp_growth_pct",
    "education_growth_pct",
    "urban_growth_pct"
]

df_cluster = df[cluster_vars].apply(pd.to_numeric, errors="coerce").replace([np.inf, -np.inf], np.nan).dropna()

scaler = StandardScaler()
# Rebuild the clustering dataset without discarding every incomplete row
cluster_data = (
    df[cluster_vars]
    .apply(pd.to_numeric, errors="coerce")
    .replace([np.inf, -np.inf], np.nan)
)

# Remove variables containing no usable observations
usable_vars = cluster_data.columns[cluster_data.notna().any()].tolist()

if not usable_vars:
    raise ValueError(
        "No hay datos numéricos disponibles para el clustering. "
        "Revisa las columnas de energía y las uniones del dataset."
    )

df_cluster = cluster_data[usable_vars].dropna(how="all")

# Impute missing values using each variable's median
imputer = SimpleImputer(strategy="median")
df_cluster = pd.DataFrame(
    imputer.fit_transform(df_cluster),
    index=df_cluster.index,
    columns=usable_vars
)

if len(df_cluster) < 4:
    raise ValueError(
        f"No hay suficientes observaciones para 4 clusters: "
        f"{len(df_cluster)} filas disponibles."
    )

df_scaled = scaler.fit_transform(df_cluster)

kmeans = KMeans(n_clusters=4, random_state=42)
clusters = kmeans.fit_predict(df_scaled)

df["cluster_socio_energy"] = None
df.loc[df_cluster.index, "cluster_socio_energy"] = clusters

cluster_labels = {
    0: "Alta transición + Alto desarrollo",
    1: "Transición media + Desarrollo medio",
    2: "Transición baja + Desarrollo alto",
    3: "Transición baja + Desarrollo bajo"
}

df["cluster_socio_energy_label"] = df["cluster_socio_energy"].map(cluster_labels)

# ------------------------------------------------------------
# 10. ANÁLISIS ESTADÍSTICO
# ------------------------------------------------------------

corr_by_country = (
    df.groupby("country")
      .apply(lambda x: x[
          [
              "gdp",
              "gdp_per_capita_constant_2015",
              "school_enrollment_tertiary_pct",
              "urban_population_pct",
              "unemployment_rate",
              "labor_force_participation_rate",
              "SDI",
              "UMI",
              "LEI",
              "renewable_share"
          ]
      ].corr()["renewable_share"].dropna())
      .reset_index()
)

reg_df = df.dropna(subset=[
    "renewable_share",
    "gdp_per_capita_constant_2015",
    "school_enrollment_tertiary_pct",
    "urban_population_pct",
    "labor_force_participation_rate",
    "unemployment_rate"
])

reg_df["interaction_gdp_edu"] = (
    reg_df["gdp_per_capita_constant_2015"] *
    reg_df["school_enrollment_tertiary_pct"]
)

Y = reg_df["renewable_share"]
X = reg_df[
    [
        "gdp_per_capita_constant_2015",
        "school_enrollment_tertiary_pct",
        "urban_population_pct",
        "labor_force_participation_rate",
        "unemployment_rate",
        "interaction_gdp_edu",
        "SDI",
        "UMI",
        "LEI"
    ]
]
X = sm.add_constant(X)

model = sm.OLS(Y, X).fit()

reg_summary = pd.DataFrame({
    "variable": model.params.index,
    "coef": model.params.values,
    "std_err": model.bse.values,
    "t": model.tvalues.values,
    "p_value": model.pvalues.values,
    "ci_lower": model.conf_int()[0].values,
    "ci_upper": model.conf_int()[1].values
})

# ------------------------------------------------------------
# 11. GLOSARIO
# ------------------------------------------------------------

glosario = pd.DataFrame([
    {"métrica": "solar_share_energy", "definición": "Porcentaje de energía solar.", "interpretación": "Indica adopción de energía solar."},
    {"métrica": "wind_share_energy", "definición": "Porcentaje de energía eólica.", "interpretación": "Indica adopción de energía eólica."},
    {"métrica": "hydro_share_energy", "definición": "Porcentaje de energía hidroeléctrica.", "interpretación": "Indica dependencia de hidro."},
    {"métrica": "biofuel_share_energy", "definición": "Porcentaje de bioenergía.", "interpretación": "Indica uso de biomasa."},
    {"métrica": "nuclear_share_energy", "definición": "Porcentaje de energía nuclear.", "interpretación": "Indica dependencia nuclear."},
    {"métrica": "fossil_share_energy", "definición": "Porcentaje de energía fósil.", "interpretación": "Indica dependencia de combustibles fósiles."},
    {"métrica": "low_carbon_share_energy", "definición": "Porcentaje de energía baja en carbono.", "interpretación": "Indica avance hacia descarbonización."},
])

# ------------------------------------------------------------
# 12. EXPORTACIÓN A EXCEL
# ------------------------------------------------------------

fact_powerbi = df.copy()

output_path = Path(carpeta) / OUTPUT_FILE

with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
    glosario.to_excel(writer, sheet_name="glosario", index=False)
    corr_by_country.to_excel(writer, sheet_name="corr_by_country", index=False)
    reg_summary.to_excel(writer, sheet_name="regression_summary", index=False)
    fact_powerbi.to_excel(writer, sheet_name="fact_powerbi", index=False)

print("Archivo generado correctamente:", output_path)


ValueError: Found array with 0 sample(s) (shape=(0, 14)) while a minimum of 1 is required by StandardScaler.